# Notebook 2.3: The Husimi Q-Function and Heterodyne Measurement
### Module 2: Phase Space and Quasiprobability Distributions
**Computational Quantum Optics Curriculum**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_2_phase_space_quasiprobabilities/03_husimi_q_function.ipynb)

---

### Prerequisites
- Notebook 2.1: The Wigner Quasiprobability Distribution Fundamentals
- Notebook 2.2: Wigner Negativity as a Metric for Quantum Non-Classicality
- Notebook 1.2: Coherent States, Displacement, and Poissonian Statistics

### Pedagogical Learning Objectives
1. **Mathematical Definition of the Husimi $Q$-Function:** Formulate $Q(\alpha) = \frac{1}{\pi}\langle \alpha | \hat{\rho} | \alpha \rangle$ and understand why it is bounded ($0 \le Q(\alpha) \le 1/\pi$) and unconditionally non-negative.
2. **The Weierstrass Transform:** Understand how the Husimi distribution emerges as the Gaussian convolution (half-quantum vacuum smoothing) of the Wigner function.
3. **Balanced Optical Heterodyne Detection (8-Port Homodyne):** Model the physical measurement apparatus that directly measures $Q(\alpha)$, connecting simultaneous quadrature detection to the Arthurs-Kelly minimum uncertainty bound.
4. **Stellar Representation & Zeros of $Q$:** Explore the zeros of $Q(\alpha)$ as a unique fingerprint of pure quantum states.
5. **Wehrl Phase-Space Entropy:** Formulate Alfred Wehrl's information-theoretic entropy $S_Q = -\int Q \ln Q \, d^2\alpha$, verifying Elliott Lieb's theorem ($S_Q \ge 1$) with minimum entropy achieved by coherent states.
6. **Numerical Synthesis & Wigner vs. Husimi Comparison:** Implement vectorized numerical solvers and contrast $W(x, p)$ with $Q(x, p)$ across canonical quantum states.


In [ ]:
# Setup, environment detection, and library imports
import os
import sys
import math
import numpy as np
import scipy.linalg as la
from scipy.special import genlaguerre
import matplotlib.pyplot as plt

# Ensure matplotlib handles headless environments gracefully
if 'MPLBACKEND' not in os.environ:
    os.environ['MPLBACKEND'] = 'Agg' if not hasattr(sys, 'ps1') else 'module://matplotlib_inline.backend_inline'

# Plot styling configuration for high-DPI publication graphics
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['lines.linewidth'] = 2.0
plt.rcParams['grid.alpha'] = 0.35

# Detect QuTiP availability with clean fallback
try:
    import qutip as qt
    HAS_QUTIP = True
    print(f"QuTiP {qt.__version__} successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

# Detect interactive widgets environment
try:
    from ipywidgets import interact, FloatSlider, Dropdown
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


## 1. Definition and Non-Negativity of the Husimi $Q$-Function

In 1940, Japanese physicist **Kôdi Husimi** introduced a phase-space distribution defined via the diagonal matrix elements of the density operator in the overcomplete coherent state basis (*Proceedings of the Physico-Mathematical Society of Japan* 22, 264, 1940):

$$
Q(\alpha) \equiv \frac{1}{\pi} \langle \alpha | \hat{\rho} | \alpha \rangle
$$

where $|\alpha\rangle$ is a standard Glauber coherent state with complex eigenvalue $\alpha = \frac{x + ip}{\sqrt{2}}$.

---

### Unconditional Non-Negativity and Boundedness

Unlike the Wigner function $W(x, p)$, which can take negative values:
1. **Strict Positivity:**
   Because any valid density matrix $\hat{\rho}$ is positive semidefinite ($\hat{\rho} \ge 0$), the expectation value in any state $|\alpha\rangle$ is strictly non-negative:
   $$
   Q(\alpha) \ge 0 \quad \text{for all } \alpha \in \mathbb{C}
   $$
2. **Absolute Upper Bound:**
   By Cauchy-Schwarz inequality, $|\langle \alpha | \psi \rangle|^2 \le \langle \alpha | \alpha \rangle \langle \psi | \psi \rangle = 1$. Therefore:
   $$
   0 \le Q(\alpha) \le \frac{1}{\pi} \approx 0.3183
   $$
   The upper bound $1/\pi$ is saturated *if and only if* the field is in the exact pure coherent state $|\alpha_0\rangle$, evaluated at the phase-space point $\alpha = \alpha_0$.
3. **Unit Normalization:**
   Using the resolution of identity for coherent states $\frac{1}{\pi} \int |\alpha\rangle\langle \alpha| \, d^2\alpha = \hat{I}$:
   $$
   \int Q(\alpha) \, d^2\alpha = \frac{1}{\pi} \int \langle \alpha | \hat{\rho} | \alpha \rangle \, d^2\alpha = \mathrm{Tr}\left( \hat{\rho} \, \frac{1}{\pi} \int |\alpha\rangle\langle \alpha| \, d^2\alpha \right) = \mathrm{Tr}(\hat{\rho}) = 1
   $$

In dimensionless quadrature coordinates $(x, p)$ where $d^2\alpha = \frac{dx \, dp}{2}$:
$$
Q_{\text{quad}}(x, p) = \frac{1}{2} Q(\alpha) = \frac{1}{2\pi} \langle \alpha | \hat{\rho} | \alpha \rangle \quad \Longrightarrow \quad \iint_{-\infty}^\infty Q_{\text{quad}}(x, p) \, dx \, dp = 1
$$


## 2. The Weierstrass Transform: Wigner Smoothing

What is the relationship between the Wigner quasiprobability distribution $W(x, p)$ and the Husimi $Q$-distribution?

The Husimi function is the **Weierstrass transform (Gaussian convolution)** of the Wigner function:

$$
Q(\alpha) = \frac{2}{\pi} \iint_{\mathbb{C}} W(\beta) \, e^{-2|\alpha - \beta|^2} \, d^2\beta
$$

In quadrature coordinates:

$$
Q_{\text{quad}}(x, p) = \frac{1}{\pi} \iint_{-\infty}^\infty W(x', p') \, \exp\left[ -(x - x')^2 - (p - p')^2 \right] \, dx' \, dp'
$$

---

### Physical Mechanism: The Added Quantum of Vacuum Noise

Notice the variance of the Gaussian kernel:
$$
\sigma^2 = \frac{1}{2}
$$
This corresponds precisely to **one quantum of zero-point vacuum fluctuation**!

- The Wigner function represents a phase-space distribution with **zero added measurement noise**, which is why quantum interference ripples and negative lobes are fully preserved.
- The Husimi function convolves the Wigner function with an isotropic Gaussian of width $\sigma = 1/\sqrt{2}$. This convolution completely smooths out rapid quantum oscillations and obliterates all negative regions, rendering $Q(x, p) \ge 0$ everywhere!

**The Classical Price:** While $Q(x, p)$ can be interpreted as a bona fide classical probability density, it achieves this at the cost of **resolution**: quantum features smaller than the vacuum noise ellipse are irreversibly blurred.


In [ ]:
# Comprehensive Numerical Solver for the Husimi Q-Function

def compute_husimi_grid(state, xvec, pvec):
    """Computes Husimi Q-distribution Q(x, p) on a 2D Cartesian grid from state vector or density matrix."""
    if state.ndim == 1:
        psi = state / np.linalg.norm(state)
        rho = np.outer(psi, psi.conj())
    else:
        rho = state / np.real(np.trace(state))
        
    N = rho.shape[0]
    X, P = np.meshgrid(xvec, pvec)
    alpha = (X + 1j * P) / np.sqrt(2.0)
    
    # Precompute normalization factor 1 / sqrt(n!)
    fact = np.array([float(math.factorial(n)) for n in range(N)])
    
    # Vector of basis functions V_n(alpha) = (alpha*)^n / sqrt(n!)
    M_p, M_x = len(pvec), len(xvec)
    V = np.zeros((N, M_p, M_x), dtype=np.complex128)
    for n in range(N):
        V[n] = (np.conj(alpha)**n) / np.sqrt(fact[n])
        
    if state.ndim == 1:
        # Pure state projection: <alpha|psi> = exp(-|alpha|^2 / 2) sum_n c_n V_n
        psi_proj = np.tensordot(psi, V, axes=(0, 0))
        Q_unscaled = np.abs(psi_proj)**2
    else:
        # Density matrix: sum_{n, m} rho_{n, m} V_m* V_n
        V_flat = V.reshape(N, -1)
        rho_V = rho @ V_flat
        Q_flat = np.sum(np.conj(V_flat) * rho_V, axis=0)
        Q_unscaled = np.real(Q_flat.reshape(M_p, M_x))
        
    # Scale to quadrature normalization: Q_quad(x, p) = (1 / 2pi) * exp(-|alpha|^2) * Q_unscaled
    Q_quad = (1.0 / (2.0 * np.pi)) * np.exp(-np.abs(alpha)**2) * Q_unscaled
    return Q_quad

# Verification benchmark: Vacuum |0> and Single-Photon Fock |1>
N_bench = 20
x_eval = np.linspace(-4.5, 4.5, 91)
p_eval = np.linspace(-4.5, 4.5, 91)
dx = x_eval[1] - x_eval[0]
dp = p_eval[1] - p_eval[0]

# 1. Vacuum State |0>
vac_bench = np.zeros(N_bench, dtype=np.complex128); vac_bench[0] = 1.0
Q_vac = compute_husimi_grid(vac_bench, x_eval, p_eval)

# 2. Single-Photon Fock State |1>
fock1_bench = np.zeros(N_bench, dtype=np.complex128); fock1_bench[1] = 1.0
Q_fock1 = compute_husimi_grid(fock1_bench, x_eval, p_eval)

# Diagnostics
norm_q_vac = np.sum(Q_vac) * dx * dp
norm_q_fock1 = np.sum(Q_fock1) * dx * dp
center_idx = len(x_eval) // 2

print("--- Husimi Q-Function Numerical Benchmark ---")
print(f"Vacuum Peak Value Q(0,0):     {Q_vac[center_idx, center_idx]:.6f} | Expected: {1.0/(2.0*np.pi):.6f}")
print(f"Vacuum Integral over dx dp:   {norm_q_vac:.6f} | Expected: 1.000000")
print(f"Fock |1> Center Value Q(0,0): {Q_fock1[center_idx, center_idx]:.6f} | Expected: 0.000000 (Stellar Zero!)")
print(f"Fock |1> Minimum Value:       {np.min(Q_fock1):.6e} | Expected: >= 0.0")
print(f"Fock |1> Integral over dx dp: {norm_q_fock1:.6f} | Expected: 1.000000")

assert abs(norm_q_vac - 1.0) < 1e-3, "Vacuum Q normalization failed!"
assert abs(norm_q_fock1 - 1.0) < 1e-3, "Fock |1> Q normalization failed!"
assert np.min(Q_fock1) >= -1e-15, "Husimi Q must be non-negative!"
print("Husimi Q-function verified: strictly non-negative, unit normalized, and accurate!")


## 3. Optical Heterodyne Detection (8-Port Homodyne)

How is the Husimi $Q$-function measured experimentally?

In **balanced optical homodyne detection**, a signal field is mixed with a strong local oscillator to measure a single quadrature $\hat{x}_\theta$ with zero added noise.

To measure **both non-commuting quadratures $\hat{x}$ and $\hat{p}$ simultaneously**, one employs an **8-port homodyne interferometer (heterodyne detection)**:

```
                      Vacuum Ancilla |0⟩_v
                              │
                              ▼
Signal Mode â ───► [ 50:50 Beam Splitter ]
                         /          \
                        /            \
                       ▼              ▼
                 Port 1 (ĉ)        Port 2 (d̂)
                     │                 │
                     ▼                 ▼
             [Homodyne of x̂_c]   [Homodyne of p̂_d]
```

---

### The Arthurs-Kelly Theorem (1965)

The two output modes after the beam splitter are:
$$
\hat{c} = \frac{\hat{a} + \hat{v}}{\sqrt{2}}, \qquad \hat{d} = \frac{\hat{a} - \hat{v}}{\sqrt{2}}
$$
Because $\hat{c}$ and $\hat{d}$ are spatially separated physical modes, their quadratures commute:
$$
\left[ \hat{x}_c, \hat{p}_d \right] = 0
$$
They can be measured simultaneously without quantum backaction!

However, mapping these measurement outcomes back to the input mode $\hat{a}$:
$$
\hat{x}_{\text{meas}} = \sqrt{2} \hat{x}_c = \hat{x}_a + \hat{x}_v
$$
$$
\hat{p}_{\text{meas}} = \sqrt{2} \hat{p}_d = \hat{p}_a - \hat{p}_v
$$

The vacuum ancilla port injects zero-point fluctuations:
$$
\langle (\Delta \hat{x}_v)^2 \rangle = \frac{1}{2}, \qquad \langle (\Delta \hat{p}_v)^2 \rangle = \frac{1}{2}
$$
According to the **Arthurs-Kelly theorem** (*Bell System Technical Journal* 44, 725, 1965), simultaneously measuring two conjugate quantum observables costs at least one quantum of noise ($\hbar$).

**The Fundamental Connection:**
The joint probability distribution of heterodyne measurement outcomes $(x_{\text{meas}}, p_{\text{meas}})$ is **identically the Husimi $Q$-function**:
$$
P_{\text{heterodyne}}(x_{\text{meas}}, p_{\text{meas}}) \equiv Q_{\text{quad}}(x, p)
$$


In [ ]:
# Comparative Phase Space Gallery: Wigner W(x, p) vs. Husimi Q(x, p)

# Helper function to compute Wigner on same grid
def compute_wigner_quick(state, xvec, pvec):
    if state.ndim == 1:
        psi = state / np.linalg.norm(state)
        rho = np.outer(psi, psi.conj())
    else:
        rho = state / np.real(np.trace(state))
    N = rho.shape[0]
    X, P = np.meshgrid(xvec, pvec)
    alpha = (X + 1j * P) / np.sqrt(2.0)
    z = 4.0 * np.abs(alpha)**2
    exp_factor = (1.0 / np.pi) * np.exp(-2.0 * np.abs(alpha)**2)
    W = np.zeros_like(X, dtype=np.complex128)
    fact = np.array([float(math.factorial(n)) for n in range(N)])
    for n in range(N):
        if abs(rho[n, n]) > 1e-15:
            W += rho[n, n] * ((-1)**n) * genlaguerre(n, 0)(z)
        for m in range(n + 1, N):
            if abs(rho[n, m]) > 1e-15:
                d = m - n
                coeff = ((-1)**n) * np.sqrt(fact[n] / fact[m]) * ((2.0 * np.conj(alpha))**d)
                term = coeff * genlaguerre(n, d)(z)
                W += rho[n, m] * term + rho[m, n] * np.conj(term)
    return np.real(W) * exp_factor

# 1. Single-Photon Fock state |1>
W_f1 = compute_wigner_quick(fock1_bench, x_eval, p_eval)
Q_f1 = Q_fock1

# 2. Squeezed Vacuum state |0, r=0.7>
a_op = np.diag(np.sqrt(np.arange(1, N_bench)), k=1)
r_param = 0.7
S_op = la.expm(0.5 * r_param * (a_op@a_op - a_op.T.conj()@a_op.T.conj()))
psi_sq = S_op @ vac_bench
W_sq = compute_wigner_quick(psi_sq, x_eval, p_eval)
Q_sq = compute_husimi_grid(psi_sq, x_eval, p_eval)

# Plotting: Side-by-side comparison (2x2 grid)
fig, axes = plt.subplots(2, 2, figsize=(13, 11))

# Panel (0,0): Wigner of Fock |1>
cf00 = axes[0, 0].contourf(x_eval, p_eval, W_f1, levels=np.linspace(-0.35, 0.35, 45), cmap='RdBu_r', extend='both')
axes[0, 0].contour(x_eval, p_eval, W_f1, levels=[0.0], colors='black', linewidths=1.8, linestyles='--')
axes[0, 0].set_title(r"Wigner $W(x, p)$: Fock $|1\rangle$ (Negative Core)")
plt.colorbar(cf00, ax=axes[0, 0], fraction=0.046, pad=0.04)

# Panel (0,1): Husimi of Fock |1>
cf01 = axes[0, 1].contourf(x_eval, p_eval, Q_f1, levels=np.linspace(0, 0.12, 45), cmap='viridis')
axes[0, 1].set_title(r"Husimi $Q(x, p)$: Fock $|1\rangle$ (Smoothed, $Q \geq 0$)")
plt.colorbar(cf01, ax=axes[0, 1], fraction=0.046, pad=0.04)

# Panel (1,0): Wigner of Squeezed Vacuum
cf10 = axes[1, 0].contourf(x_eval, p_eval, W_sq, levels=np.linspace(-0.05, 0.35, 45), cmap='RdBu_r', extend='both')
axes[1, 0].set_title(rf"Wigner $W(x, p)$: Squeezed Vacuum ($r={r_param}$)")
plt.colorbar(cf10, ax=axes[1, 0], fraction=0.046, pad=0.04)

# Panel (1,1): Husimi of Squeezed Vacuum
cf11 = axes[1, 1].contourf(x_eval, p_eval, Q_sq, levels=np.linspace(0, 0.12, 45), cmap='viridis')
axes[1, 1].set_title(rf"Husimi $Q(x, p)$: Squeezed Vacuum ($r={r_param}$)")
plt.colorbar(cf11, ax=axes[1, 1], fraction=0.046, pad=0.04)

for ax in axes.flat:
    ax.set_aspect('equal')
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_xlabel(r"Quadrature $x$")
    ax.set_ylabel(r"Quadrature $p$")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. The Wehrl Phase-Space Entropy

Because the Husimi function $Q(\alpha)$ is a non-negative, normalized probability density, one can define its **Shannon information entropy**, introduced by Austrian physicist **Alfred Wehrl** in 1979 (*Reports on Mathematical Physics* 16, 353, 1979):

$$
S_Q(\hat{\rho}) \equiv - \iint_{\mathbb{C}} Q(\alpha) \ln \left[ \pi Q(\alpha) \right] \, d^2\alpha
$$

---

### Elliott Lieb's Theorem (1978)

Wehrl conjectured, and mathematical physicist **Elliott H. Lieb** famously proved (*Communications in Mathematical Physics* 62, 35, 1978):

$$
S_Q(\hat{\rho}) \ge 1
$$

The minimum phase-space entropy bound $S_Q = 1$ is **uniquely saturated if and only if $\hat{\rho} = |\alpha\rangle\langle \alpha|$ is a pure coherent state**! (Note: if defined with bare $\ln Q(\alpha)$ without the $\pi$ scaling, the bound is $1 + \ln\pi \approx 2.1447$).

### Physical Interpretation:
- The von Neumann entropy $S_{\text{vN}}(\hat{\rho}) = -\mathrm{Tr}(\hat{\rho}\ln\hat{\rho})$ vanishes for any pure state ($S_{\text{vN}}(|n\rangle) = 0$).
- In contrast, the Wehrl entropy measures the **information content in continuous phase space**. Because of quantum vacuum noise, no state can have zero Wehrl entropy. The bound $S_Q \ge 1$ represents the irreducible phase-space uncertainty of the quantum vacuum!
- For Fock states $|n\rangle$:
  $$
  S_Q(|n\rangle) = 1 + n - \sum_{k=1}^n \frac{1}{k} + \ln(n!) \approx 1 + \ln(n+1)
  $$
  Wehrl entropy increases with photon number, reflecting the expanding phase-space ring.


In [ ]:
# Computing Wehrl Entropy S_Q across Canonical Quantum States

def wehrl_entropy(Q_grid, xvec, pvec):
    """Computes Wehrl phase-space entropy S_Q = - int Q(alpha) ln Q(alpha) d^2 alpha."""
    # Note: d^2 alpha = (dx * dp) / 2
    # Q(alpha) = 2 * Q_quad(x, p)
    Q_alpha = 2.0 * Q_grid
    d2_alpha = (xvec[1] - xvec[0]) * (pvec[1] - pvec[0]) / 2.0
    
    # Filter strictly positive values to avoid log(0)
    pos_mask = Q_alpha > 1e-15
    integrand = Q_alpha[pos_mask] * np.log(np.pi * Q_alpha[pos_mask])
    s_q = -np.sum(integrand) * d2_alpha
    return s_q

# Compute Wehrl entropy for Fock states n = 0, 1, 2, 3
x_fine = np.linspace(-5.5, 5.5, 131)
p_fine = np.linspace(-5.5, 5.5, 131)

wehrl_fock = []
for n_val in range(4):
    psi_n = np.zeros(N_bench, dtype=np.complex128)
    psi_n[n_val] = 1.0
    Q_n = compute_husimi_grid(psi_n, x_fine, p_fine)
    s_val = wehrl_entropy(Q_n, x_fine, p_fine)
    wehrl_fock.append(s_val)

print("--- Wehrl Phase-Space Entropy S_Q Verification ---")
print(f"Vacuum State |0>: S_Q = {wehrl_fock[0]:.4f} (Lieb Bound: 1.0000)")
print(f"Fock State |1>:   S_Q = {wehrl_fock[1]:.4f} (Expected: 1 + 1 - 1 = 1.0 + ln(e) = 2.0000 - ...)")
print(f"Fock State |2>:   S_Q = {wehrl_fock[2]:.4f}")
print(f"Fock State |3>:   S_Q = {wehrl_fock[3]:.4f}")

assert abs(wehrl_fock[0] - 1.0) < 1e-2, "Vacuum Wehrl entropy test failed!"
assert all(wehrl_fock[i] < wehrl_fock[i+1] for i in range(len(wehrl_fock)-1)), "Wehrl entropy should increase with n!"
print("Wehrl entropy verified: minimum saturated at S_Q = 1 for vacuum, monotonically growing with n!")


In [ ]:
# Interactive Exploration: Wigner-to-Husimi Continuous Smoothing

def explore_wigner_to_husimi(state_type='Fock |1>', smooth_sigma=0.0):
    """Demonstrates continuous Gaussian smoothing from Wigner (sigma=0) to Husimi (sigma=1/sqrt(2))."""
    from scipy.ndimage import gaussian_filter
    
    N_calc = 15
    x_int = np.linspace(-3.5, 3.5, 71)
    p_int = np.linspace(-3.5, 3.5, 71)
    dx_val = x_int[1] - x_int[0]
    
    if state_type == 'Fock |1>':
        psi = np.zeros(N_calc, dtype=np.complex128); psi[1] = 1.0
    elif state_type == 'Fock |2>':
        psi = np.zeros(N_calc, dtype=np.complex128); psi[2] = 1.0
    elif state_type == 'Cat State':
        alpha_c = 1.8
        a = np.diag(np.sqrt(np.arange(1, N_calc)), k=1)
        D1 = la.expm(alpha_c * a.T.conj() - alpha_c * a)
        D2 = la.expm(-alpha_c * a.T.conj() + alpha_c * a)
        vac = np.zeros(N_calc, dtype=np.complex128); vac[0] = 1.0
        psi = D1 @ vac + D2 @ vac
        psi /= np.linalg.norm(psi)
    else:  # Squeezed
        a = np.diag(np.sqrt(np.arange(1, N_calc)), k=1)
        S = la.expm(0.5 * 0.7 * (a@a - a.T.conj()@a.T.conj()))
        vac = np.zeros(N_calc, dtype=np.complex128); vac[0] = 1.0
        psi = S @ vac
        
    W_raw = compute_wigner_quick(psi, x_int, p_int)
    
    # Gaussian filter smoothing
    if smooth_sigma > 0.01:
        pixel_sigma = smooth_sigma / dx_val
        W_smoothed = gaussian_filter(W_raw, sigma=pixel_sigma)
    else:
        W_smoothed = W_raw.copy()
        
    min_val = np.min(W_smoothed)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # 2D contour
    v_limit = max(0.25, np.max(np.abs(W_smoothed)))
    cf = ax1.contourf(x_int, p_int, W_smoothed, levels=np.linspace(-v_limit, v_limit, 40),
                      cmap='RdBu_r', extend='both')
    ax1.contour(x_int, p_int, W_smoothed, levels=[0.0], colors='black', linewidths=1.8, linestyles='--')
    ax1.set_aspect('equal')
    ax1.set_xlabel("x")
    ax1.set_ylabel("p")
    ax1.set_title(rf"{state_type} (Smoothing $\sigma={smooth_sigma:.2f}$, Min $W={min_val:+.3f}$)")
    plt.colorbar(cf, ax=ax1, fraction=0.046, pad=0.04)
    
    # 1D slice along x
    center_p = len(p_int) // 2
    slice_1d = W_smoothed[center_p, :]
    ax2.plot(x_int, slice_1d, 'b-', lw=2.2, label=r'Cross-section at $p=0$')
    ax2.axhline(0.0, color='black', lw=1.2, ls='--')
    ax2.fill_between(x_int, slice_1d, 0, where=(slice_1d < 0), color='red', alpha=0.3, label='Negative Quasiprobability')
    ax2.set_xlim(-3.5, 3.5)
    ax2.set_xlabel("x")
    ax2.set_ylabel("Amplitude")
    ax2.set_title(f"Negative Region Disappearance (Min: {min_val:+.4f})")
    ax2.legend(loc='upper right')
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

# Launch interactive exploration
if HAS_WIDGETS:
    interact(explore_wigner_to_husimi,
             state_type=Dropdown(options=['Fock |1>', 'Fock |2>', 'Cat State', 'Squeezed'], value='Fock |1>'),
             smooth_sigma=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.0, description=r'Filter \sigma:'))
else:
    # Static execution fallback for headless environments
    explore_wigner_to_husimi(state_type='Fock |1>', smooth_sigma=0.0)


## 5. Self-Check Exercises and Pedagogical Challenges

Consolidate your mastery of the Husimi $Q$-distribution and phase-space measurement theory through these analytical and numerical challenges.

---

### Challenge 1: Analytical Derivation of $Q_n(\alpha)$ for Fock State $|n\rangle$
1. Using the definition $Q_n(\alpha) = \frac{1}{\pi}|\langle \alpha | n \rangle|^2$ and the coherent state expansion $|\alpha\rangle = e^{-|\alpha|^2/2}\sum_{m=0}^\infty \frac{\alpha^m}{\sqrt{m!}}|m\rangle$, show that:
   $$
   Q_n(\alpha) = \frac{1}{\pi n!} |\alpha|^{2n} e^{-|\alpha|^2}
   $$
2. In polar coordinates ($r = |\alpha|$), find the radius $r_{\text{max}}$ where $Q_n(\alpha)$ reaches its maximum value. Explain why this matches the classical circular orbit of energy $E = \hbar\omega(n + 1/2)$.

<details>
<summary><b>Click to expand Solution to Challenge 1</b></summary>

**Part 1 Solution:**
The overlap between coherent state $|\alpha\rangle$ and Fock state $|n\rangle$ is:
$$
\langle \alpha | n \rangle = e^{-|\alpha|^2/2} \sum_{m=0}^\infty \frac{(\alpha^*)^m}{\sqrt{m!}} \langle m | n \rangle = e^{-|\alpha|^2/2} \frac{(\alpha^*)^n}{\sqrt{n!}}
$$
Taking the absolute square and dividing by $\pi$:
$$
Q_n(\alpha) = \frac{1}{\pi} |\langle \alpha | n \rangle|^2 = \frac{1}{\pi} e^{-|\alpha|^2} \frac{|\alpha|^{2n}}{n!} \quad \blacksquare
$$

**Part 2 Solution:**
Let $u = |\alpha|^2 = r^2$. We maximize $f(u) = u^n e^{-u}$:
$$
\frac{df}{du} = n u^{n-1} e^{-u} - u^n e^{-u} = u^{n-1} e^{-u} (n - u) = 0 \quad \Longrightarrow \quad u_{\text{max}} = n
$$
Therefore:
$$
r_{\text{max}} = \sqrt{n}
$$
In dimensionless quadrature coordinates $(x, p)$ where $r_{\text{quad}} = \sqrt{x^2 + p^2} = \sqrt{2}|\alpha|$:
$$
r_{\text{quad, max}} = \sqrt{2n}
$$
This ring of radius $\sqrt{2n}$ corresponds directly to the classical circular trajectory in phase space having average photon number $\bar{n} = n$!
</details>

---

### Challenge 2: Numerical Verification with QuTiP
Using QuTiP, compute the Husimi $Q$-distribution for a coherent state with $\alpha = 1.5$ using `qutip.qfunc()`, and verify that the peak value is $1/\pi \approx 0.3183$ and that $Q$ is strictly non-negative everywhere.

<details>
<summary><b>Click to expand Solution to Challenge 2</b></summary>

```python
if HAS_QUTIP:
    import qutip as qt
    N = 25
    alpha_0 = 1.5
    psi_coh = qt.coherent(N, alpha_0)
    
    x = np.linspace(-3, 5, 80)
    p = np.linspace(-3, 3, 80)
    Q_qt = qt.qfunc(psi_coh, x, p)
    
    print(f"QuTiP Q minimum: {np.min(Q_qt):.6e}")
    print(f"QuTiP Q maximum: {np.max(Q_qt):.6f}")
    assert np.min(Q_qt) >= 0.0, "Husimi Q must be non-negative!"
    print("QuTiP validation confirmed Husimi Q positivity!")
else:
    print("QuTiP not installed; run in Google Colab to execute this test.")
```
</details>


## 6. Summary, Reference Table, and Outlook

### Comprehensive Comparison: Wigner $W(x, p)$ vs. Husimi $Q(x, p)$

| Property | Wigner Function $W(x, p)$ | Husimi $Q$-Function $Q(\alpha)$ |
| :--- | :--- | :--- |
| **Operator Basis** | Parity operator $\hat{\Pi}(\alpha)$ | Projector onto coherent state $|\alpha\rangle\langle \alpha|$ |
| **Operator Ordering** | Symmetric (Weyl) ordering | Anti-normal ordering ($\hat{a}$ before $\hat{a}^\dagger$) |
| **Positivity** | Can be negative ($W < 0$ for non-Gaussian states) | **Strictly non-negative everywhere ($0 \le Q \le 1/\pi$)** |
| **Added Measurement Noise** | Zero ($0$) | One quantum of vacuum noise ($\hbar$) |
| **Direct Physical Measurement** | Homodyne Tomography (Radon backprojection) | **8-Port Homodyne (Balanced Heterodyne Detection)** |
| **Information Entropy** | Undefined (due to negativity) | **Wehrl Entropy $S_Q \ge 1$** (Lieb Bound) |
| **Phase Space Resolution** | Maximum quantum resolution | Coarse-grained over vacuum ellipse ($\sigma = 1/\sqrt{2}$) |

---

### Key Takeaways
1. **The Non-Negative Phase Space:** $Q(\alpha) = \frac{1}{\pi}\langle \alpha|\hat{\rho}|\alpha\rangle$ is an unconditionally non-negative, bounded probability density describing the quantum state.
2. **Weierstrass Smoothing:** $Q$ is the Gaussian convolution of $W$ with a vacuum noise kernel of width $1/\sqrt{2}$, which washes away all negative quasiprobabilities.
3. **Heterodyne Equivalence:** Simultaneous balanced measurement of conjugate quadratures (8-port homodyne detection) directly measures $Q(x, p)$ in accordance with the Arthurs-Kelly minimum noise theorem.

---

### Primary References
- **Husimi, K.** (1940). *Some formal properties of the density matrix*. Proceedings of the Physico-Mathematical Society of Japan. 3rd Series, 22(4), 264–314.
- **Arthurs, E., & Kelly, J. L.** (1965). *On the simultaneous measurement of a pair of conjugate observables*. Bell System Technical Journal, 44(4), 725–729.
- **Wehrl, A.** (1979). *On the relation between classical and quantum-mechanical entropy*. Reports on Mathematical Physics, 16(3), 353–358.
- **Lieb, E. H.** (1978). *Proof of an entropy conjecture of Wehrl*. Communications in Mathematical Physics, 62(1), 35–41.

---

### Next in the Curriculum
In **Notebook 2.4: The Glauber-Sudarshan P-Representation**, we explore the opposite extreme of operator ordering: the singular, highly non-classical $P(\alpha)$ distribution corresponding to normal ordering, completing the holy trinity of quantum phase-space quasiprobabilities ($P, W, Q$).
